# Week 3: Train Contrastive Probe from Labeled Data

## Purpose

This notebook loads the **manually labeled** training data and trains the contrastive probe.

## Input

`training_data_labeled.csv` with columns:
- `prompt`: Base prompt
- `next_token`: Actual token model predicts
- `full_text`: prompt + next_token
- `label`: **'code'** or **'language'** (manually labeled)

## Process

1. Load labeled data
2. Extract hidden states for each `full_text`
3. Train logistic regression probe
4. Test with contrastive generation

---

In [2]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [3]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import entropy as scipy_entropy
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports complete")

✅ Imports complete


In [4]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()

print(f"✅ Model loaded on {model.device}")

Loading codellama/CodeLlama-7b-Instruct-hf...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Model loaded on cuda:0


In [5]:
# Cell 4: Load labeled data

LABELED_DATA_FILE = '/content/training_data_labeled.csv'

print(f"Loading {LABELED_DATA_FILE}...")
df = pd.read_csv(LABELED_DATA_FILE)

print(f"\n✅ Loaded {len(df)} training examples")
print(f"\nDataset info:")
print(df.info())

# Check labels
print(f"\nLabel distribution:")
print(df['label'].value_counts())

# Validate labels
valid_labels = df['label'].isin(['code', 'language'])
if not valid_labels.all():
    print(f"\n⚠️  WARNING: Found {(~valid_labels).sum()} rows with invalid labels!")
    print(f"Invalid labels: {df[~valid_labels]['label'].unique()}")
    print(f"\nRemoving invalid rows...")
    df = df[valid_labels]
    print(f"Remaining: {len(df)} examples")

# Convert to binary labels
df['label_binary'] = df['label'].map({'language': 0, 'code': 1})

print(f"\nBinary label distribution:")
print(f"  LANGUAGE (0): {(df['label_binary']==0).sum()}")
print(f"  CODE (1): {(df['label_binary']==1).sum()}")

print(f"\nSample entries:")
print(df[['full_text', 'label', 'probability']].head(10))

Loading /content/training_data_labeled.csv...

✅ Loaded 800 training examples

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   prompt         800 non-null    object 
 1   expected_type  800 non-null    object 
 2   next_token     787 non-null    object 
 3   probability    800 non-null    float64
 4   full_text      800 non-null    object 
 5   label          800 non-null    object 
dtypes: float64(1), object(5)
memory usage: 37.6+ KB
None

Label distribution:
label
language    675
code        125
Name: count, dtype: int64

Binary label distribution:
  LANGUAGE (0): 675
  CODE (1): 125

Sample entries:
                                full_text     label  probability
0     The authentication is done usingthe  language     0.235474
1       The authentication is done usinga  language     0.158081
2       The authentication is do

In [6]:
# Cell 5: Extract hidden states

SELECTED_LAYERS = [8, 16, 31]

def get_multi_layer_state(text: str, layers: List[int]) -> np.ndarray:
    """Extract and concatenate hidden states from multiple layers."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    return np.concatenate(states).astype(np.float32)

print(f"Extracting hidden states from layers {SELECTED_LAYERS}...")
print(f"This may take a few minutes for {len(df)} examples...\n")

X_train = []
y_train = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
    h = get_multi_layer_state(row['full_text'], SELECTED_LAYERS)
    X_train.append(h)
    y_train.append(row['label_binary'])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"\n✅ Hidden states extracted")
print(f"   Shape: {X_train.shape}")
print(f"   Feature dimension: {X_train.shape[1]:,} (3 layers × 4096)")

Extracting hidden states from layers [8, 16, 31]...
This may take a few minutes for 800 examples...



Extracting:   0%|          | 0/800 [00:00<?, ?it/s]


✅ Hidden states extracted
   Shape: (800, 12288)
   Feature dimension: 12,288 (3 layers × 4096)


In [7]:
# Cell 6: Train probe

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

probe = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(probe, X_train_scaled, y_train, cv=cv)

cv_accuracy = accuracy_score(y_train, y_pred_cv)

print(f"\n{'='*80}")
print(f"PROBE TRAINING RESULTS")
print(f"{'='*80}")
print(f"\nLayers: {SELECTED_LAYERS}")
print(f"Training examples: {len(y_train)}")
print(f"5-Fold CV Accuracy: {cv_accuracy:.1%}")

print(f"\n{classification_report(y_train, y_pred_cv, target_names=['LANGUAGE (0)', 'CODE (1)'])}")

# Confusion matrix
cm = confusion_matrix(y_train, y_pred_cv)
print(f"Confusion Matrix:")
print(f"                Pred LANG  Pred CODE")
print(f"True LANG          {cm[0,0]:>4}       {cm[0,1]:>4}")
print(f"True CODE          {cm[1,0]:>4}       {cm[1,1]:>4}")

# Train final model
probe.fit(X_train_scaled, y_train)
print(f"\n✅ Final probe trained on all data")


PROBE TRAINING RESULTS

Layers: [8, 16, 31]
Training examples: 800
5-Fold CV Accuracy: 97.0%

              precision    recall  f1-score   support

LANGUAGE (0)       0.99      0.97      0.98       675
    CODE (1)       0.87      0.95      0.91       125

    accuracy                           0.97       800
   macro avg       0.93      0.96      0.95       800
weighted avg       0.97      0.97      0.97       800

Confusion Matrix:
                Pred LANG  Pred CODE
True LANG           657         18
True CODE             6        119

✅ Final probe trained on all data


# Contrastive Generation Testing

In [8]:
# Cell 7: Helper functions

def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def entropy_from_probs(probs: np.ndarray) -> float:
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

def classify_token_type(text: str) -> Tuple[int, float]:
    """Classify: 1=code token, 0=language token."""
    h = get_multi_layer_state(text, SELECTED_LAYERS).reshape(1, -1)
    h_scaled = scaler.transform(h)
    token_type = probe.predict(h_scaled)[0]
    probability = probe.predict_proba(h_scaled)[0, 1]
    return int(token_type), float(probability)

def analyze_candidate_tokens(
    prompt: str,
    top_k: int = 10,
    verbose: bool = False
) -> Dict:
    """
    Get top-K candidate next tokens and classify each as CODE or LANGUAGE.
    Returns majority vote and detailed breakdown.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()

    probs = softmax(logits)
    top_indices = np.argsort(probs)[-top_k:][::-1]

    candidates = []
    code_votes = 0
    lang_votes = 0

    for idx in top_indices:
        token = tokenizer.decode([idx])
        prob = probs[idx]

        # Classify prompt + candidate token
        completion = prompt + token
        token_type, type_prob = classify_token_type(completion)

        candidates.append({
            'token': token,
            'prob': prob,
            'type': 'CODE' if token_type == 1 else 'LANGUAGE',
            'type_prob': type_prob
        })

        if token_type == 1:
            code_votes += prob
        else:
            lang_votes += prob

    is_code_uncertainty = code_votes > lang_votes

    if verbose:
        print(f"\nCandidate analysis:")
        for c in candidates:
            print(f"  '{c['token']}' (p={c['prob']:.3f}) → {c['type']} (conf={c['type_prob']:.3f})")
        print(f"\nVotes: CODE={code_votes:.3f}, LANGUAGE={lang_votes:.3f}")
        print(f"Decision: {'CODE uncertainty' if is_code_uncertainty else 'LANGUAGE uncertainty'}")

    return {
        'candidates': candidates,
        'code_votes': code_votes,
        'lang_votes': lang_votes,
        'is_code_uncertainty': is_code_uncertainty,
        'confidence': max(code_votes, lang_votes) / (code_votes + lang_votes) if (code_votes + lang_votes) > 0 else 0
    }

print("✅ Helper functions ready")

✅ Helper functions ready


In [ ]:
# Cell 8: Contrastive generation function

def generate_with_contrastive_probe(
    prompt: str,
    entropy_threshold: float = 3.0,
    top_k_candidates: int = 10,
    max_tokens: int = 50,
    verbose: bool = True
) -> Dict:
    """
    Contrastive entropy-driven generation.

    At each step:
    1. Generate next token
    2. Compute entropy H
    3. If H > threshold:
       - Get top-K candidate next tokens
       - Classify each "prompt + candidate" as CODE or LANGUAGE
       - Weighted vote: If majority CODE → STOP
       - If majority LANGUAGE → Continue
    4. If H ≤ threshold: Continue (confident)
    """
    current_text = prompt
    generated_token_ids = []  # Track token IDs for proper decoding
    entropy_trace = []
    stop_reason = None
    stop_info = {}

    if verbose:
        print(f"\n{'='*80}")
        print(f"Prompt: '{prompt}'")
        print(f"Entropy threshold: {entropy_threshold:.1f} bits")
        print(f"{'='*80}")

    for step in range(max_tokens):
        inputs = tokenizer(current_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0, -1, :].cpu().numpy()

        probs = softmax(logits)
        H = entropy_from_probs(probs)
        entropy_trace.append(H)

        next_token_id = np.argmax(probs)
        next_token = tokenizer.decode([next_token_id])

        if verbose:
            print(f"\nStep {step + 1}: '{next_token}' H={H:.2f}")

        if H > entropy_threshold:
            if verbose:
                print(f"  ⚠️  HIGH ENTROPY - analyzing candidates...")

            analysis = analyze_candidate_tokens(
                current_text,
                top_k=top_k_candidates,
                verbose=verbose
            )

            if analysis['is_code_uncertainty']:
                if verbose:
                    print(f"  ❗ CODE UNCERTAINTY - STOPPING!")
                stop_reason = "code_uncertainty"
                stop_info = {
                    'step': step,
                    'entropy': H,
                    'candidates': analysis['candidates'],
                    'code_votes': analysis['code_votes'],
                    'lang_votes': analysis['lang_votes']
                }
                break
            else:
                if verbose:
                    print(f"  ✓ LANGUAGE uncertainty - continuing")

        generated_token_ids.append(next_token_id)
        current_text += next_token

        if next_token_id == tokenizer.eos_token_id:
            stop_reason = "eos"
            break

    if stop_reason is None:
        stop_reason = "max_tokens"

    # Properly decode the full sequence of generated tokens
    generated_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True)

    if verbose:
        print(f"\n{'='*80}")
        print(f"Stop: {stop_reason}")
        print(f"Generated: '{generated_text}'")
        print(f"Full: '{prompt}{generated_text}'")
        print(f"{'='*80}")

    return {
        'prompt': prompt,
        'generated_text': generated_text,
        'full_text': prompt + generated_text,
        'entropy_trace': entropy_trace,
        'stop_reason': stop_reason,
        'stop_info': stop_info,
        'num_steps': len(generated_token_ids)
    }

print("✅ Generation function ready")

# Testing

In [10]:
# Cell 9: Demo test

print("\n" + "="*80)
print("DEMO: Testing Contrastive Probe")
print("="*80)

test_prompts = [
    "The authentication is done using",  # Should STOP (code uncertainty)
    "The authentication is",              # Should CONTINUE (language uncertainty)
]

for test_prompt in test_prompts:
    result = generate_with_contrastive_probe(
        test_prompt,
        entropy_threshold=3.0,
        top_k_candidates=10,
        max_tokens=5,
        verbose=True
    )
    print("\n" + "-"*80 + "\n")


DEMO: Testing Contrastive Probe

Prompt: 'The authentication is done using'
Entropy threshold: 3.0 bits

Step 1: 'the' H=5.88
  ⚠️  HIGH ENTROPY - analyzing candidates...

Candidate analysis:
  'the' (p=0.235) → LANGUAGE (conf=0.000)
  'a' (p=0.158) → LANGUAGE (conf=0.000)
  'O' (p=0.053) → CODE (conf=0.999)
  'an' (p=0.046) → LANGUAGE (conf=0.000)
  '[' (p=0.040) → CODE (conf=1.000)
  '`' (p=0.029) → CODE (conf=1.000)
  'J' (p=0.025) → CODE (conf=1.000)
  'Open' (p=0.015) → CODE (conf=1.000)
  'HTTP' (p=0.014) → CODE (conf=1.000)
  'Spring' (p=0.009) → CODE (conf=1.000)

Votes: CODE=0.183, LANGUAGE=0.439
Decision: LANGUAGE uncertainty
  ✓ LANGUAGE uncertainty - continuing

Step 2: '`' H=8.73
  ⚠️  HIGH ENTROPY - analyzing candidates...

Candidate analysis:
  '`' (p=0.070) → CODE (conf=0.999)
  'following' (p=0.062) → LANGUAGE (conf=0.001)
  'O' (p=0.038) → CODE (conf=0.991)
  '[' (p=0.033) → CODE (conf=0.995)
  'J' (p=0.018) → CODE (conf=1.000)
  'user' (p=0.014) → CODE (conf=0.996)


In [ ]:
# Cell 10: Full Sentence Generation Testing

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# ============================================================================
# TEST CASES - FULL SENTENCE GENERATION
# ============================================================================
# IMPROVED: Added context to CODE cases, added code-like words to LANGUAGE cases

# CODE test cases - we EXPECT the system to stop at least once for code uncertainty
# IMPROVEMENT: Added technical context to make it clear we're talking about implementation
CODE_TEST_CASES = [
    # Authentication & Security (with context)
    {'prompt': 'In our React app, authentication is done using', 'category': 'auth_method'},
    {'prompt': 'In the backend, passwords are hashed with', 'category': 'auth_hash'},
    {'prompt': 'For our API, JWT tokens are signed using', 'category': 'auth_signing'},
    {'prompt': 'In production, the OAuth provider we use is', 'category': 'auth_provider'},
    {'prompt': 'On the server, session data is stored in', 'category': 'session_store'},

    # Databases (with context)
    {'prompt': 'For data persistence, the database we use is', 'category': 'database_type'},
    {'prompt': 'In the application, we query the database using', 'category': 'database_query'},
    {'prompt': 'For database access, the ORM library is', 'category': 'database_orm'},
    {'prompt': 'To improve performance, caching is implemented with', 'category': 'database_cache'},

    # Web Frameworks (with context)
    {'prompt': 'For the REST API, the framework we use is', 'category': 'web_framework'},
    {'prompt': 'In production, the web server runs on', 'category': 'web_server'},
    {'prompt': 'In the client code, HTTP requests are made using', 'category': 'http_client'},
    {'prompt': 'For data fetching, our GraphQL server uses', 'category': 'graphql_server'},

    # Frontend (with context)
    {'prompt': 'For the UI, the frontend framework is', 'category': 'frontend_framework'},
    {'prompt': 'In the application, state management is handled by', 'category': 'frontend_state'},
    {'prompt': 'For the interface, components are built with', 'category': 'frontend_components'},
    {'prompt': 'In the SPA, routing is done using', 'category': 'frontend_routing'},

    # ML/AI (with context)
    {'prompt': 'For training, the model is trained with', 'category': 'ml_framework'},
    {'prompt': 'In our neural network, deep learning is implemented using', 'category': 'ml_deep_learning'},
    {'prompt': 'For gradient descent, the optimizer we use is', 'category': 'ml_optimizer'},

    # DevOps & Cloud (with context)
    {'prompt': 'For hosting, we deploy to', 'category': 'cloud_platform'},
    {'prompt': 'In Kubernetes, containers are orchestrated with', 'category': 'cloud_containers'},
    {'prompt': 'For automation, the CI/CD pipeline uses', 'category': 'cloud_cicd'},

    # Testing & Build (with context)
    {'prompt': 'In the test suite, unit tests are written with', 'category': 'test_unit'},
    {'prompt': 'For building assets, the bundler we use is', 'category': 'build_bundler'},
    {'prompt': 'For dependencies, package management is done with', 'category': 'build_package_manager'},
]

# LANGUAGE test cases - we EXPECT the system to NEVER stop (only language uncertainty)
# IMPROVEMENT: Added examples with code-like words but in non-technical contexts
LANGUAGE_TEST_CASES = [
    # Pure descriptions (no code references)
    {'prompt': 'The weather today is', 'category': 'description_weather'},
    {'prompt': 'The meeting yesterday was', 'category': 'description_meeting'},
    {'prompt': 'My favorite color has always been', 'category': 'description_color'},
    {'prompt': 'The book I read last week was', 'category': 'description_book'},
    {'prompt': 'The movie we watched seemed', 'category': 'description_movie'},

    # General explanations
    {'prompt': 'The main idea of the story is to', 'category': 'explanation_idea'},
    {'prompt': 'The cooking process works by', 'category': 'explanation_process'},
    {'prompt': 'This teaching approach helps to', 'category': 'explanation_approach'},
    {'prompt': 'The benefit of exercise is', 'category': 'explanation_benefit'},

    # Instructions with CODE-LIKE WORDS but non-technical context
    {'prompt': 'When installing furniture in my home, you should', 'category': 'instruction_furniture'},
    {'prompt': 'To debug a relationship problem, first', 'category': 'instruction_debug'},
    {'prompt': 'Before deploying troops, the general needs to', 'category': 'instruction_deploy'},
    {'prompt': 'The configuration of the room requires', 'category': 'instruction_config'},
    {'prompt': 'To optimize your morning routine, try', 'category': 'instruction_optimize'},

    # More non-technical uses of technical words
    {'prompt': 'The framework of the argument is', 'category': 'nontechnical_framework'},
    {'prompt': 'My mental state is managed by', 'category': 'nontechnical_state'},
    {'prompt': 'The library in town has', 'category': 'nontechnical_library'},
    {'prompt': 'Running the business takes', 'category': 'nontechnical_running'},
    {'prompt': 'The function of the heart is', 'category': 'nontechnical_function'},
    {'prompt': 'Implementing the new policy will', 'category': 'nontechnical_implement'},
]

ALL_TEST_CASES = [
    {**case, 'expected_stopped': True} for case in CODE_TEST_CASES  # Should stop at least once
] + [
    {**case, 'expected_stopped': False} for case in LANGUAGE_TEST_CASES  # Should never stop
]

print(f"\n{'='*80}")
print(f"FULL SENTENCE GENERATION TESTING")
print(f"{'='*80}")
print(f"\nTotal test cases: {len(ALL_TEST_CASES)}")
print(f"  CODE tests (should stop): {len(CODE_TEST_CASES)}")
print(f"  LANGUAGE tests (should not stop): {len(LANGUAGE_TEST_CASES)}")
print(f"\nIMPROVEMENTS:")
print(f"  ✓ Added technical context to CODE prompts")
print(f"  ✓ Added code-like words to LANGUAGE prompts (non-technical context)")

# ============================================================================
# RUN FULL SENTENCE GENERATION TESTS
# ============================================================================

print(f"\n{'='*80}")
print(f"RUNNING FULL SENTENCE GENERATION")
print(f"{'='*80}\n")

test_results = []

for test_case in tqdm(ALL_TEST_CASES, desc="Testing"):
    prompt = test_case['prompt']

    # Generate full sentence (up to period or max tokens)
    result = generate_with_contrastive_probe(
        prompt,
        entropy_threshold=3.0,
        top_k_candidates=10,
        max_tokens=20,  # Generate longer sequences
        verbose=False
    )

    # Check if it stopped for code uncertainty at ANY point during generation
    stopped_for_code = result['stop_reason'] == 'code_uncertainty'
    expected_stopped = test_case['expected_stopped']
    correct = stopped_for_code == expected_stopped

    # Get max entropy across all generation steps
    max_entropy = np.max(result['entropy_trace']) if result['entropy_trace'] else 0

    test_results.append({
        'prompt': prompt,
        'category': test_case['category'],
        'expected_stopped': expected_stopped,
        'stopped_for_code': stopped_for_code,
        'correct': correct,
        'stop_reason': result['stop_reason'],
        'max_entropy': max_entropy,
        'generated_text': result.get('generated_text', ''),
        'num_tokens_generated': result.get('num_steps', 0),  # FIXED: use num_steps
    })

print(f"\n✅ Testing complete on {len(test_results)} cases!")

# ============================================================================
# DETAILED ANALYSIS
# ============================================================================

df_results = pd.DataFrame(test_results)

print(f"\n{'='*80}")
print(f"TEST RESULTS - FULL SENTENCE GENERATION")
print(f"{'='*80}")

overall_accuracy = df_results['correct'].mean()
print(f"\n📊 OVERALL ACCURACY: {overall_accuracy:.1%}")
print(f"   (on {len(df_results)} test cases)")

print(f"\n📈 BY CLASS:")
for expected_val in [True, False]:
    class_name = "CODE (should stop)" if expected_val else "LANGUAGE (should not stop)"
    subset = df_results[df_results['expected_stopped'] == expected_val]
    accuracy = subset['correct'].mean() if len(subset) > 0 else 0
    correct = subset['correct'].sum()
    total = len(subset)
    stopped_count = subset['stopped_for_code'].sum()
    print(f"\n   {class_name}")
    print(f"      Correct: {correct}/{total}")
    print(f"      Accuracy: {accuracy:.1%}")
    print(f"      Stopped for code: {stopped_count}/{total}")
    if total > 0:
        print(f"      Avg entropy: {subset['max_entropy'].mean():.2f} bits")
        print(f"      Avg tokens generated: {subset['num_tokens_generated'].mean():.1f}")

# Confusion matrix
tp = len(df_results[(df_results['expected_stopped']==True) & (df_results['stopped_for_code']==True)])
fp = len(df_results[(df_results['expected_stopped']==False) & (df_results['stopped_for_code']==True)])
fn = len(df_results[(df_results['expected_stopped']==True) & (df_results['stopped_for_code']==False)])
tn = len(df_results[(df_results['expected_stopped']==False) & (df_results['stopped_for_code']==False)])

print(f"\n🎯 CONFUSION MATRIX")
print(f"                       Predicted NO STOP    Predicted STOPPED")
print(f"Expected NO STOP             {tn:<12}          {fp:<12}")
print(f"Expected STOP                {fn:<12}          {tp:<12}")

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n📐 METRICS (CODE class - should stop)")
print(f"   Precision: {precision:.1%}")
print(f"   Recall: {recall:.1%}")
print(f"   F1-Score: {f1:.1%}")

# Category breakdown
print(f"\n📋 PERFORMANCE BY CATEGORY")
print(f"{'Category':<30} {'Correct':<10} {'Stopped':<10} {'Accuracy':<12} {'Avg Tokens'}")
print(f"{'-'*85}")
for category in sorted(df_results['category'].unique()):
    cat_data = df_results[df_results['category'] == category]
    correct = cat_data['correct'].sum()
    stopped = cat_data['stopped_for_code'].sum()
    total = len(cat_data)
    accuracy = correct / total if total > 0 else 0
    avg_tokens = cat_data['num_tokens_generated'].mean()
    print(f"{category:<30} {correct}/{total:<8} {stopped}/{total:<8} {accuracy:>6.1%}        {avg_tokens:>6.1f}")

# Errors with generated text
errors = df_results[~df_results['correct']]
if len(errors) > 0:
    print(f"\n❌ ERRORS ({len(errors)}/{len(df_results)}):")
    for idx, error in errors.iterrows():
        exp = "SHOULD STOP" if error['expected_stopped'] else "SHOULD NOT STOP"
        act = "STOPPED" if error['stopped_for_code'] else "DID NOT STOP"
        print(f"\n   Prompt: '{error['prompt']}'")
        print(f"      Expected: {exp}, Actual: {act}")
        print(f"      Generated: '{error['generated_text']}'")
        print(f"      Stop reason: {error['stop_reason']}, Max entropy: {error['max_entropy']:.2f}")
else:
    print(f"\n🎉 PERFECT! No errors!")

print(f"\n{'='*80}")